# Speech Recognition

Companion notebook for the [Speech Recognition lesson](https://ml-viz-ruby.vercel.app/courses/speech-audio/02-speech-recognition).

We implement the **CTC collapse rule** (merge repeats, drop blanks) that turns a per-frame character
stream into text, count the **alignments** that map to a target, and do a greedy CTC decode. Pure
NumPy — the alignment logic at the heart of modern ASR.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
BLANK = '_'    # the CTC blank token

## 1 — The CTC collapse rule

Collapse a frame-level labeling to text: first merge consecutive repeats, then remove blanks. The
blank is what lets 'hello' (with a real double-l) survive.

In [ ]:
def ctc_collapse(path):
    out = []
    prev = None
    for c in path:
        if c != prev:        # merge consecutive repeats
            out.append(c)
        prev = c
    return ''.join(c for c in out if c != BLANK)   # then drop blanks

for path in ['hhh_e_lll_llo', 'h_e_l_l_o', 'hello']:
    print(f'{path:15s} -> "{ctc_collapse(path)}"')
print('\nNote the blank between the two l\'s is what preserves the double-l in "hello".')

## 2 — Many alignments map to one target

CTC sums the probability over *every* frame-labeling that collapses to the target. We enumerate the
alignments of a short target over a few frames to see how many there are.

In [ ]:
from itertools import product

def count_alignments(target, T, vocab):
    n = 0
    for path in product(vocab, repeat=T):
        if ctc_collapse(path) == target:
            n += 1
    return n

vocab = ['a', 'b', BLANK]
for T in [3, 4, 5, 6]:
    print(f'T={T} frames: {count_alignments("ab", T, vocab)} alignments collapse to "ab"')
print('\nThe count grows fast -> CTC sums them with dynamic programming, never enumerating.')

## 3 — Greedy CTC decoding

The simplest decode: at each frame take the most likely token, then collapse. We make a toy
per-frame probability matrix and decode it.

In [ ]:
vocab2 = ['c', 'a', 't', BLANK]
# per-frame probabilities (rows = frames, cols = vocab) peaking at c,a,a,t,blank,t
frames = ['c', 'a', 'a', 't', BLANK, 't']
probs = np.full((len(frames), len(vocab2)), 0.05)
for i, ch in enumerate(frames):
    probs[i, vocab2.index(ch)] = 0.85

def greedy_decode(probs, vocab):
    path = ''.join(vocab[i] for i in probs.argmax(axis=1))
    return ctc_collapse(path), path

text, path = greedy_decode(probs, vocab2)
print(f'argmax path: {path}')
print(f'decoded:     "{text}"')

## ✏️ Your turn

**Exercise.** Implement `collapse(path, blank)` (the general CTC collapse: merge repeats then drop
the blank symbol) and `is_valid_alignment(path, target, blank)` (True if the path collapses exactly
to the target). These define what CTC sums over.

In [ ]:
def collapse(path, blank='_'):
    # TODO(you): merge consecutive duplicates, then remove the blank token
    return ...

def is_valid_alignment(path, target, blank='_'):
    # TODO(you): True if collapse(path) equals target
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert collapse('hh_e_ll_llo') == 'hello'        # blank preserves the double-l
assert collapse('aa_aa') == 'aa'                  # blank separates two a-groups
assert collapse('aaaa') == 'a'                    # no blank -> repeats merge to one
assert is_valid_alignment('c_aa_t', 'cat')
assert not is_valid_alignment('caat', 'caat')    # collapses to 'cat', not 'caat'
print('\u2713 CTC collapse and alignment check are correct')

<details>
<summary>Solution</summary>

```python
def collapse(path, blank='_'):
    out, prev = [], None
    for c in path:
        if c != prev:
            out.append(c)
        prev = c
    return ''.join(c for c in out if c != blank)

def is_valid_alignment(path, target, blank='_'):
    return collapse(path, blank) == target
```

The collapse rule is the whole reason CTC needs no frame-level labels: the model can emit any
alignment that collapses to the target, and training sums their probabilities via dynamic
programming.

</details>